# CRYSTALS-Kyber με Πλήρη Μαθηματική Ανάλυση

**Μετα-Κβαντικός Μηχανισμός Ενθυλάκωσης Κλειδιού | ML-KEM / FIPS 203**

---

Αυτό το notebook διδάσκει τα **πλήρη μαθηματικά** του αλγορίθμου CRYSTALS-Kyber.
Κάθε έννοια εξηγείται πρώτα και στη συνέχεια υλοποιείται αμέσως σε Python.

**Παράμετροι παιχνιδιού** που έχουν ίδια δομή με το FIPS 203, απλώς μικρότεροι αριθμοί:

| Σύμβολο | Τιμή | Σημασία |
|--------|-------|---------|
| q | 17 | Πρώτος αριθμός mod, όλοι οι συντελεστές ανήκουν στο {0 … 16} |
| n | 8 | Βαθμός πολυωνύμου, modulus δακτυλίου είναι X⁸ + 1 |
| k | 2 | Διάσταση module, ο A είναι 2×2, τα διανύσματα έχουν 2 πολυώνυμα |
| η | 2 | Όριο θορύβου CBD, μυστικοί συντελεστές στο {−2,−1,0,1,2} |

**Πώς να χρησιμοποιήσετε αυτό το notebook:**
1. Εκτελέστε πρώτα το **Module 0: Καθολική Εγκατάσταση**. Ορίζει όλες τις κοινές σταθερές.
2. Ακολουθήστε τα modules με τη σειρά. Κάθε κελί βασίζεται στο προηγούμενο.
3. Όλες οι πράξεις γίνονται mod **q = 17** εκτός αν αναφέρεται διαφορετικά.

---
> **Όλοι οι αριθμοί σε αυτό το notebook έχουν επαληθευτεί από το έγγραφο αναφοράς CRYSTALS-Kyber.**

---
## Module 0: Καθολική Εγκατάσταση

> **Εκτελέστε αυτό το κελί πρώτα.** Όλα τα άλλα κελιά εξαρτώνται από αυτές τις σταθερές.
> Μην τροποποιήσετε τις τιμές παρακάτω. Είναι η επαληθευμένη βάση αναφοράς για όλες τις ασκήσεις.

Οι σταθερές χωρίζονται σε τρεις ομάδες:
- **Παράμετροι** που αποτελούν τις μαθηματικές ρυθμίσεις αυτής της έκδοσης Kyber
- **Τιμές Δημιουργίας Κλειδιού** μυστικό κλειδί s, σφάλμα e, ρίζες NTT, δημόσιο κλειδί t
- **Τιμές Ενθυλάκωσης και Αποενθυλάκωσης** κρυπτοκείμενο u, v και ανάκτηση w

In [ ]:
# ΚΑΘΟΛΙΚΗ ΕΓΚΑΤΑΣΤΑΣΗ: εκτελέστε πρώτα, μην τροποποιήσετε

# Παράμετροι παιχνιδιού:
Q     = 17   # πρώτος αριθμός mod
N     = 8    # βαθμός πολυωνύμου
K     = 2    # διάσταση module
ETA   = 2    # παράμετρος θορύβου CBD eta
N_INV = 15   # πολλαπλασιαστικό αντίστροφο του n:  8 × 15 = 120 ≡ 1 mod 17

# Ρίζες NTT
# roots[k]     = ω^k  mod 17   όπου ω = 9
# inv_roots[k] = (ω⁻¹)^k mod 17  όπου ω⁻¹ = 2
ROOTS     = [1, 9, 13, 15, 16, 8, 4, 2]
INV_ROOTS = [1, 2,  4,  8, 16, 15, 13, 9]

# Πίνακας A (χώρος NTT, παράγεται από δημόσιο σπόρο ρ)
A = [
    [[2,11,8,1,15,3,3,6],  [6,6,9,9,12,12,15,15]],
    [[7,14,3,10,5,12,1,8], [3,7,2,9,14,4,11,6  ]]
]

# Μυστικό κλειδί s και σφάλμα e (έξοδος CBD, κανονικός χώρος)
S0 = [1, 0, 0, 1, 0, 0, 0, 1]   # από bytes [0x01, 0x10, 0x00, 0x10]
S1 = [0, 1, 0, 0, 1, 0, 0, 0]   # από bytes [0x10, 0x00, 0x01, 0x00]
E0 = [1, 0, 0, 0, 0, 0, 0, 0]   # από bytes [0x01, 0x00, 0x00, 0x00]
E1 = [0, 0, 1, 0, 0, 0, 0, 0]   # από bytes [0x00, 0x01, 0x00, 0x00]

# Μορφές των s και e στον χώρο NTT
S0_NTT = [3,  1,  9,  1, 16,  1, 10,  1]
S1_NTT = [2,  8, 14, 14,  0,  7,  5,  1]
E0_NTT = [1,  1,  1,  1,  1,  1,  1,  1]
E1_NTT = [1, 13, 16,  4,  1, 13, 16,  4]

# Δημόσιο κλειδί t (χώρος NTT)
T0 = [2,  9, 12, 9,  3,  3,  4,  5]
T1 = [11, 15,  3, 4, 13,  2, 13,  1]

# Ενθυλάκωση: νέος θόρυβος και μήνυμα
R0    = [1, 0, 1, 0, 0, 1, 0, 0]
R1    = [0, 1, 0, 0, 1, 0, 1, 0]
E1_0  = [0, 0, 0, 1, 0, 0, 0, 0]
E1_1  = [0, 0, 0, 0, 0, 1, 0, 0]
E2    = [0, 0, 0, 0, 1, 0, 0, 0]
R0_NTT = [3,  5, 13,  7,  1,  6,  4,  3]
R1_NTT = [3, 12, 13, 10,  1, 11,  4, 14]
MSG   = [1, 0, 1, 0, 1, 0, 1, 0]
M_HAT = [8, 0, 8, 0, 8, 0, 8, 0]   # κωδικοποιημένο μήνυμα: bit→8 αν 1, bit→0 αν 0

# Κρυπτοκείμενο
U0 = [0, 15,  3, 12,  9, 16,  3,  4]
U1 = [8,  7,  5,  8, 16, 12,  6,  0]
V  = [6,  6,  6,  9, 14, 12,  9, 10]

# Ενδιάμεσες τιμές αποενθυλάκωσης
U0_NTT = [11,  0, 11, 14,  2, 16, 12,  2]
U1_NTT = [11,  3,  3,  2,  8,  6,  6,  8]
STU_N  = [13,  7, 15,  9,  5, 11,  2, 10]
W      = [10, 16,  8,  0,  9,  1,  7,  0]
DECODED= [ 1,  0,  1,  0,  1,  0,  1,  0]

print(f"Εγκατάσταση ολοκληρώθηκε.  Q={Q}  N={N}  K={K}  ETA={ETA}  N_INV={N_INV}")
print(f"Ευθείες ρίζες:   {ROOTS}")
print(f"Αντίστροφες ρίζες:   {INV_ROOTS}")

---
## Module 1: Οι Μαθηματικές Βάσεις

### 1.1 Ο Δακτύλιος R_q = Z₁₇[X] / (X⁸ + 1)

Όλη η αριθμητική του Kyber γίνεται μέσα σε αυτόν τον δακτύλιο:
- **Κάθε πολυώνυμο έχει το πολύ 8 όρους**, δηλαδή συντελεστές από X⁰ έως X⁷
- **Όλοι οι συντελεστές είναι ακέραιοι mod q = 17** και ανήκουν στο {0, 1, …, 16}
- **Αρνητική κυκλική αναγωγή:** όταν ο πολλαπλασιασμός παράγει X^k με k ≥ 8, χρησιμοποιούμε την ταυτότητα
 `X⁸ ≡ −1 mod (X⁸+1)`, που δίνει:
 `X⁸ → −1`, `X⁹ → −X`, `X¹⁰ → −X²`, …, `X¹⁵ → −X⁷`

**Παράδειγμα:** ένας συντελεστής στο X¹⁰ μετασχηματίζεται ως `−1 × (συντελεστής του X²) mod 17`

---

### 1.2 Εύρεση της Πρωτόγονης Ρίζας g

Χρειαζόμαστε μια γεννήτρια g του Z*₁₇, δηλαδή ένα στοιχείο του οποίου οι δυνάμεις παράγουν και τα 16 μη μηδενικά υπόλοιπα mod 17.

**Έλεγχος:** το g πρέπει να έχει **τάξη 16**, δηλαδή g^16 ≡ 1 mod 17 ΚΑΙ g^k ≠ 1 για κάθε 1 ≤ k < 16.
Ισοδύναμα (αφού 16 = 2⁴, μοναδικός πρώτος παράγοντας είναι το 2): ελέγχουμε ότι g^8 ≠ 1 mod 17.

- g = 2 **αποτυγχάνει**: 2⁸ mod 17 = 1 → τάξη μόνο 8, όχι 16
- g = 3 **λειτουργεί**: 3⁸ mod 17 = 16 = −1 ≠ 1 → τάξη 16 ✓

---

### 1.3 Υπολογισμός των ψ, ω και των πινάκων ριζών

Από το g = 3 παράγουμε όλα όσα χρειαζόμαστε για το NTT:

```
ψ = g^((q−1)/(2n)) mod q = 3^((17−1)/(2×8)) = 3¹ mod 17 = 3
 Check: ψ^16 = 1 mod 17 ✓ (primitive 2n-th root of unity)
 Check: ψ^8 = 16 = −1 mod 17 ✓ (must not equal 1)

ω = ψ² mod 17 = 9
 Check: ω^8 = 1 mod 17 ✓ (primitive n-th root of unity)

ω⁻¹ = 2 because 9 × 2 = 18 ≡ 1 mod 17
n⁻¹ = 15 because 8 × 15 = 120 ≡ 1 mod 17

roots[k] = ω^k mod 17 = [1, 9, 13, 15, 16, 8, 4, 2]
inv_roots[k] = (ω⁻¹)^k mod 17 = [1, 2, 4, 8, 16, 15, 13, 9]
```

**Επαλήθευση:** `roots[k] × inv_roots[k] ≡ 1 mod 17` για κάθε k

---

### Άσκηση 1: Υλοποιήστε τη `modpow` και επαληθεύστε g, ψ, ω

Υλοποιήστε τη γρήγορη αριθμητική modular ύψωση σε δύναμη με τη μέθοδο **τετραγωνισμού-και-πολλαπλασιασμού**:

```
result = 1
while exp > 0:
 if exp is odd: result = result × base mod mod
 base = base² mod mod
 exp = exp // 2
```

Στη συνέχεια χρησιμοποιήστε την για να επαληθεύσετε την αλυσίδα g → ψ → ω → πίνακες ριζών.

In [ ]:
def modpow(base: int, exp: int, mod: int) -> int:
    """Γρήγορη modular ύψωση σε δύναμη (τετραγωνισμός-και-πολλαπλασιασμός).

    Αλγόριθμος:
        result = 1
        while exp > 0:
            αν exp περιττό:  result = result * base % mod
            base = base^2 % mod
            exp  = exp >> 1
    """
    result = 1
    b      = base % mod
    while exp > 0:
        if exp & 1:
            # TODO: πολλαπλασιάστε result επί b, mod mod
            result = ...
        # TODO: τετραγωνίστε b, mod mod
        b   = ...
        # TODO: δεξί shift exp κατά 1  (exp >>= 1)
        exp = ...
    return result

print("Λειτουργεί το g = 2;")
print(f"2^8  mod 17 = {modpow(2,  8, Q)}")
print(f"2^16 mod 17 = {modpow(2, 16, Q)}")
print("\nΛειτουργεί το g = 3;")
print("k    3^k mod 17")
elements = set()
for k in range(1, 17):
    v = modpow(3, k, Q)
    elements.add(v)
    mark = "  ← ταυτότητα, τάξη = 16 επιβεβαιώθηκε ✓" if v == 1 else ""
    print(f"  {k:2d}     {v:2d}{mark}")
print(f"\nΠαράχθηκαν {len(elements)} διαφορετικές τιμές → g = 3 ΑΠΟΔΕΚΤΟ ✓")
psi       = modpow(3, (Q-1)//(2*N), Q)
omega     = modpow(psi, 2, Q)
omega_inv = modpow(omega, Q-2, Q)
n_inv_v   = modpow(N, Q-2, Q)
print(f"\nψ={psi}  ω={omega}  ω⁻¹={omega_inv}  n⁻¹={n_inv_v}")
fwd = [modpow(omega,     k, Q) for k in range(N)]
inv = [modpow(omega_inv, k, Q) for k in range(N)]
print(f"Ευθείες ρίζες:  {fwd}")
print(f"Αντίστροφες ρίζες:  {inv}")
assert fwd == ROOTS and inv == INV_ROOTS
print("\n✓ Και οι δύο πίνακες ριζών συμφωνούν με τις καθολικές σταθερές.")


---
## Module 2 Δειγματοληψία CBD: Δημιουργία s και e

### 2.1 Γιατί μικροί συντελεστές;

Η ασφάλεια του Kyber εξαρτάται από το μυστικό κλειδί **s** και το σφάλμα **e** να έχουν **μικρούς** συντελεστές.
Ο όρος θορύβου στην αποενθυλάκωση πρέπει να ικανοποιεί `|θόρυβος| < q/4 = 4.25` για σωστή αποκρυπτογράφηση.
Το CBD με η=2 εγγυάται συντελεστές στο {−2, −1, 0, +1, +2} — ακριβώς αρκετά μικροί.

### 2.2 Ο αλγόριθμος CBD (η = 2)

Είσοδος: ροή bytes από `SHAKE256(σ ‖ counter)`, διαβάζεται **LSB-πρώτο** (bit 0 του byte πρώτο).

Για κάθε 4 bits `(b₀, b₁, b₂, b₃)`:
```
coefficient = (b₀ + b₁) − (b₂ + b₃)
```
Αυτό ακολουθεί την **κεντρική διωνυμική κατανομή** με παράμετρο η=2:
τιμές −2, −1, 0, +1, +2 με πιθανότητες 1/16, 4/16, 6/16, 4/16, 1/16.

**Αρνητικοί συντελεστές** αποθηκεύονται ως θετικός αντιπρόσωπος mod q:
`−1 → 16`, `−2 → 15`

**Ένα byte → 2 συντελεστές. Τέσσερα bytes → ένα πολυώνυμο (n=8 συντελεστές).**

### 2.3 Πλήρης ιχνηλάτηση από το έγγραφο αναφοράς

```
s[0] input bytes: [0x01, 0x10, 0x00, 0x10]

Byte 0x01 = 00000001 (LSB-first: 1,0,0,0,0,0,0,0)
 coeff[0]: (b0+b1)−(b2+b3) = (1+0)−(0+0) = +1 → stored: 1
 coeff[1]: (b4+b5)−(b6+b7) = (0+0)−(0+0) = 0 → stored: 0

Byte 0x10 = 00010000 (LSB-first: 0,0,0,0,1,0,0,0)
 coeff[2]: (0+0)−(0+0) = 0 → stored: 0
 coeff[3]: (1+0)−(0+0) = +1 → stored: 1

Byte 0x00 → coeff[4]=0, coeff[5]=0
Byte 0x10 → coeff[6]=0, coeff[7]=+1

s[0] = [1, 0, 0, 1, 0, 0, 0, 1] ✓
```

---

### Άσκηση 2: Υλοποιήστε τη `cbd_sample`

Μετατρέψτε μια λίστα bytes σε συντελεστές πολυωνύμου χρησιμοποιώντας τον αλγόριθμο CBD.

In [ ]:
def cbd_sample(byte_list: list, q: int) -> list:
    """CBD η=2: μετατροπή bytes σε συντελεστές πολυωνύμου.

    Για κάθε byte:
      1. bits = [(byte_val >> i) & 1  for i in range(8)]    ← LSB πρώτο
      2. Για κάθε ομάδα 4 bits (pair = 0 ή 1):
             b0, b1, b2, b3 = bits[pair*4 : pair*4+4]
             coeff = (b0 + b1) - (b2 + b3)     ← κεντρική διωνυμική
             αποθήκευση coeff % q
    """
    coefficients = []
    for byte_val in byte_list:
        bits = [(byte_val >> i) & 1 for i in range(8)]
        for pair in range(2):
            b0, b1, b2, b3 = bits[pair*4 : pair*4+4]
            # TODO: coeff = (b0 + b1) - (b2 + b3)
            coeff = ...
            # TODO: coefficients.append(coeff % q)
            coefficients.append(...)
    return coefficients

print("CBD για bytes s[0] [0x01, 0x10, 0x00, 0x10]\n")
for idx, byte_val in enumerate([0x01, 0x10, 0x00, 0x10]):
    bits = [(byte_val >> i) & 1 for i in range(8)]
    print(f"  Byte 0x{byte_val:02X}  LSB-πρώτο: {bits}")
    for pair in range(2):
        b0,b1,b2,b3=bits[pair*4:pair*4+4]; raw=(b0+b1)-(b2+b3); stored=raw%Q
        print(f"    coeff[{idx*2+pair}]: ({b0}+{b1})-({b2}+{b3})={raw:+d}  αποθ:{stored}")
    print()
s0=cbd_sample([0x01,0x10,0x00,0x10],Q); s1=cbd_sample([0x10,0x00,0x01,0x00],Q)
e0=cbd_sample([0x01,0x00,0x00,0x00],Q); e1=cbd_sample([0x00,0x01,0x00,0x00],Q)
print(f"s[0]={s0}  αναμενόμενο:{S0}")
print(f"s[1]={s1}  αναμενόμενο:{S1}")
print(f"e[0]={e0}  αναμενόμενο:{E0}")
print(f"e[1]={e1}  αναμενόμενο:{E1}")
assert s0==S0 and s1==S1 and e0==E0 and e1==E1
print("\n✓ Όλες οι έξοδοι CBD συμφωνούν με τις καθολικές σταθερές.")


---
## Module 3: Ευθύς NTT (Cooley-Tukey)

### 3.1 Γιατί NTT;

Ο απλός πολλαπλασιασμός πολυωνύμων στο R_q απαιτεί **O(n²)** πράξεις.
Ο **Θεωρητικός Μετασχηματισμός Αριθμών** μετατρέπει τα πολυώνυμα σε έναν χώρο όπου
ο πολλαπλασιασμός είναι **σημειακός** (συντελεστής × συντελεστής), μειώνοντας την πολυπλοκότητα σε **O(n log n)**.

### 3.2 Αναστροφή bits (μεταθετική)

Πριν από τα στρώματα πεταλούδας, ο πίνακας εισόδου **μεταθέτεται** με αναστροφή της δυαδικής αναπαράστασης κάθε δείκτη.

Για n=8 (δείκτες 3 bits):

| Δείκτης | Δυαδικό | Αντεστραμμένο | Νέα θέση |
|-------|--------|----------|--------------|
| 0 | 000 | 000 | 0 |
| 1 | 001 | 100 | 4 |
| 2 | 010 | 010 | 2 |
| 3 | 011 | 110 | 6 |
| 4 | 100 | 001 | 1 |
| 5 | 101 | 101 | 5 |
| 6 | 110 | 011 | 3 |
| 7 | 111 | 111 | 7 |

Τα ζεύγη (1,4) και (3,6) εναλλάσσονται. Οι θέσεις {0,2,5,7} παραμένουν.

### 3.3 Πράξη πεταλούδας (Cooley-Tukey)

Κάθε πεταλούδα παίρνει δύο τιμές **u** και **v** και έναν **παράγοντα στρέψης w**:
```
t = v × w mod q
new_u = (u + t) mod q
new_v = (u − t) mod q
```

### 3.4 Τρία στρώματα πεταλούδας για n=8

```
Layer 1: len=2, step=4 (4 blocks of 2 elements, root = roots[0] = 1)
Layer 2: len=4, step=2 (2 blocks of 4 elements)
Layer 3: len=8, step=1 (1 block of 8 elements)
```

Σε κάθε στρώμα: `step = n // len`.
Για μπλοκ που ξεκινά στο `bs`, θέσεις πεταλούδας `j` και `j + len//2`:
`w = roots[j × step]`

### 3.5 Πλήρης ιχνηλάτηση για s[0] = [1,0,0,1,0,0,0,1]

```
After bit-reversal: [1, 0, 0, 0, 0, 0, 1, 1]

Layer 1 (w=1 for all):
 [0,1]: u=1,v=0,w=1 → t=0 → [1, 1]
 [2,3]: u=0,v=0 → t=0 → [0, 0]
 [4,5]: u=0,v=0 → t=0 → [0, 0]
 [6,7]: u=1,v=1,w=1 → t=1 → [2, 0]
After layer 1: [1, 1, 0, 0, 0, 0, 2, 0]

Layer 2 (roots[0]=1, roots[2]=13):
 [0,2]: w=1 t=0 → [1, 1]
 [1,3]: w=13 t=0 → [1, 1]
 [4,6]: w=1 t=2 → [2, 15] (0−2 mod 17 = 15)
 [5,7]: w=13 t=0 → [0, 0]
After layer 2: [1, 1, 1, 1, 2, 0, 15, 0]

Layer 3 (roots[0..3] = 1, 9, 13, 15):
 [0,4]: w=1 t=2 → [3, 16]
 [1,5]: w=9 t=0 → [1, 1]
 [2,6]: w=13 t=15×13 mod 17 = 8 → [9, 10]
 [3,7]: w=15 t=0 → [1, 1]
After layer 3: [3, 1, 9, 1, 16, 1, 10, 1] ← NTT(s[0])
```

---

### Άσκηση 3: Υλοποιήστε τη `ntt`

In [ ]:
import math

def bit_reverse_permute(a: list) -> list:
    """Επιστρέφει νέα λίστα με στοιχεία αναδιατεταγμένα κατά αντεστραμμένους δείκτες bits."""
    n    = len(a)
    bits = int(math.log2(n))
    r    = list(a)
    for i in range(n):
        rev = int(f'{i:0{bits}b}'[::-1], 2)
        if rev > i:
            r[i], r[rev] = r[rev], r[i]
    return r

def ntt(poly: list, roots: list, q: int) -> list:
    """Ευθύς NTT — επαναληπτικός αλγόριθμος Cooley-Tukey.

    Βήματα:
      1. a = bit_reverse_permute(poly)
      2. length ξεκινά από 2, διπλασιάζεται (2 → 4 → 8):
             step = n // length
             για κάθε μπλοκ μεγέθους length από bs:
                 for j in range(length // 2):
                     f = bs + j
                     s = bs + j + length // 2
                     w = roots[j * step]
                     t = a[s] * w % q
                     a[f], a[s] = (a[f]+t)%q, (a[f]-t)%q
    """
    n = len(poly)
    a = bit_reverse_permute(poly)
    length = 2
    while length <= n:
        # TODO: step = n // length
        step = ...
        for bs in range(0, n, length):
            for j in range(length // 2):
                f = bs + j
                # TODO: s = bs + j + length // 2
                s = ...
                # TODO: w = roots[j * step]
                w = ...
                t = a[s] * w % q
                # TODO: butterfly:  a[f], a[s] = (a[f]+t)%q, (a[f]-t)%q
                #       Προσοχή: χρησιμοποιήστε το ΠΑΛΙΟ a[f] και για τα δύο!
                a[f], a[s] = ...
        # TODO: length <<= 1  (διπλασιάστε)
        length = ...
    return a

print("=== Πλήρης ιχνηλάτηση NTT για s[0] ===\n")
a_trace = bit_reverse_permute(list(S0))
print(f"Μετά αναστροφή bits: {a_trace}\n")
length = 2; layer = 1
while length <= N:
    step = N // length
    print(f"Layer {layer}  (len={length}, step={step}):")
    for bs in range(0, N, length):
        for j in range(length // 2):
            f,s=bs+j,bs+j+length//2; w=ROOTS[j*step]; t=a_trace[s]*w%Q; u_old=a_trace[f]
            a_trace[f]=(a_trace[f]+t)%Q; a_trace[s]=(u_old-t)%Q
            print(f"  [{f},{s}] w={w} t={t} → [{f}]={a_trace[f]} [{s}]={a_trace[s]}")
    print(f"  Μετά στρώμα {layer}: {a_trace}\n")
    length<<=1; layer+=1
print(f"NTT(s[0])={a_trace}  αναμενόμενο={S0_NTT}")
assert ntt(S0,ROOTS,Q)==S0_NTT and ntt(S1,ROOTS,Q)==S1_NTT
assert ntt(E0,ROOTS,Q)==E0_NTT and ntt(E1,ROOTS,Q)==E1_NTT
print("\n✓ Όλα τα αποτελέσματα NTT συμφωνούν με τις καθολικές σταθερές.")


---
## Module 4: Αντίστροφος NTT

### 4.1 Δομή

Ο INTT χρησιμοποιεί την **ίδια δομή πεταλούδας** με τον ευθύ NTT, με δύο διαφορές:

1. Χρησιμοποιούμε `inv_roots` αντί για `roots`
2. Μετά από όλα τα στρώματα πεταλούδας: **πολλαπλασιάζουμε κάθε συντελεστή με n⁻¹ = 15**

```
INTT(a) = scale(forward_butterfly(bit_reverse(a), inv_roots), n_inv)
```

Γι' αυτό ορίζουμε μία μόνο συνάρτηση `ntt()` και την καλούμε δύο φορές.

### 4.2 Ιχνηλάτηση κυκλικής επαλήθευσης για NTT(s[0]) = [3,1,9,1,16,1,10,1]

```
After bit-reversal: [3, 16, 9, 10, 1, 1, 1, 1]

Layer 1 (inv_roots[0]=1):
 [0,1]: t=16 → [2, 4] (3+16=19≡2; 3-16=-13≡4)
 [2,3]: t=10 → [2, 16] (9+10=19≡2; 9-10=-1≡16)
 [4,5]: t=1 → [2, 0]
 [6,7]: t=1 → [2, 0]
After layer 1: [2, 4, 2, 16, 2, 0, 2, 0]

Layer 2 (inv_roots[0]=1, inv_roots[2]=4):
 [0,2] w=1: t=2 → [4, 0]
 [1,3] w=4: t=16×4=64≡13 → 4+13=17≡0, 4-13=-9≡8 → [0, 8]
 [4,6] w=1: t=2 → [4, 0]
 [5,7] w=4: t=0 → [0, 0]
After layer 2: [4, 0, 0, 8, 4, 0, 0, 0]

Layer 3 (inv_roots[0..3] = 1, 2, 4, 8):
 [0,4] w=1: t=4 → [8, 0]
 [1,5] w=2: t=0 → [0, 0]
 [2,6] w=4: t=0 → [0, 0]
 [3,7] w=8: t=0 → [8, 0]
Before scaling: [8, 0, 0, 8, 0, 0, 0, 0]

Multiply by n⁻¹ = 15: [8×15, 0, 0, 8×15, 0, …] mod 17
 = [120 mod 17, 0, …] = [1, 0, 0, 1, 0, 0, 0, 0]
```

Περίμενε — αυτό δίνει `[1,0,0,1,0,0,0,0]` αλλά το s[0] τελειώνει με 1 στη θέση 7.
Επανέλεγχος στρώματος 3 θέση 7: a[7] προέρχεται από a[3]=8, w=8: t=0×8=0 → a[7]=8−0=8×15=1 ✓
Όντως το τελευταίο στοιχείο είναι s[0][7]=1. Πλήρες αποτέλεσμα: `[1,0,0,1,0,0,0,1]` ✓

---

### Άσκηση 4: Υλοποιήστε τη `intt`

In [ ]:
def intt(poly: list, inv_roots: list, q: int, n_inv: int) -> list:
    """Αντίστροφος NTT.

    Βήματα:
      1. Κλήση ntt(poly, inv_roots, q)   ← ίδια πεταλούδα, αντίστροφες ρίζες
      2. Κάθε συντελεστής × n_inv % q
    """
    # TODO: result = ntt(poly, inv_roots, q)
    result = ...
    # TODO: return [x * n_inv % q  for x in result]
    return [... for x in result]

print(f"INTT(NTT(s[0])) = {intt(S0_NTT, INV_ROOTS, Q, N_INV)}")
print(f"Αναμενόμενο s[0]   = {S0}")
assert intt(S0_NTT,INV_ROOTS,Q,N_INV)==S0 and intt(S1_NTT,INV_ROOTS,Q,N_INV)==S1
assert intt(E0_NTT,INV_ROOTS,Q,N_INV)==E0 and intt(E1_NTT,INV_ROOTS,Q,N_INV)==E1
print("\n✓ Όλες οι κυκλικές διαδρομές INTT συμφωνούν με τις καθολικές σταθερές.")


---
## Module 5: Δημόσιο Κλειδί: t̂ = Â·ŝ + ê

### 5.1 Γιατί όλοι οι υπολογισμοί παραμένουν στον χώρο NTT

Αφού τα s και e μετασχηματιστούν στον χώρο NTT (ŝ, ê) και το A είναι επίσης στον χώρο NTT (Â),
ο υπολογισμός του δημόσιου κλειδιού απαιτεί μόνο **σημειακές πράξεις** — δεν χρειάζεται INTT εδώ.

```
t̂[0] = Â[0][0] ⊙ ŝ[0] + Â[0][1] ⊙ ŝ[1] + ê[0]
t̂[1] = Â[1][0] ⊙ ŝ[0] + Â[1][1] ⊙ ŝ[1] + ê[1]
```

`⊙` = **σημειακός πολλαπλασιασμός** mod q (συντελεστής i × συντελεστής i)
`+` = **σημειακή πρόσθεση** mod q

### 5.2 Ιχνηλάτηση για t̂[0] — συντελεστής i=0

```
Â[0][0][0] = 2, ŝ[0][0] = 3 → 2×3 = 6
Â[0][1][0] = 6, ŝ[1][0] = 2 → 6×2 = 12
ê[0][0] = 1

t̂[0][0] = 6 + 12 + 1 = 19 mod 17 = 2 ✓
```

Και οι 8 συντελεστές του t̂[0]:

| i | Â[0][0]×ŝ[0] | Â[0][1]×ŝ[1] | +ê[0] | t̂[0][i] |
|---|--------------|--------------|-------|----------|
| 0 | 2×3=6 | 6×2=12 | 1 | (6+12+1) mod 17 = **2** |
| 1 | 11×1=11 | 6×8=48→14 | 1 | (11+14+1) mod 17 = **9** |
| 2 | 8×9=72→4 | 9×14=126→7 | 1 | (4+7+1) mod 17 = **12** |
| 3 | 1×1=1 | 9×14=7 | 1 | (1+7+1) mod 17 = **9** |
| 4 | 15×16=240→2 | 12×0=0 | 1 | (2+0+1) mod 17 = **3** |
| 5 | 3×1=3 | 12×7=84→16 | 1 | (3+16+1) mod 17 = **3** |
| 6 | 3×10=30→13 | 15×5=75→7 | 1 | (13+7+1) mod 17 = **4** |
| 7 | 6×1=6 | 15×1=15 | 1 | (6+15+1) mod 17 = **5** |

`t̂[0] = [2, 9, 12, 9, 3, 3, 4, 5]` `t̂[1] = [11, 15, 3, 4, 13, 2, 13, 1]`

---

### Άσκηση 5: Υλοποιήστε τη `keygen_t`

In [ ]:
def pointwise_mul(a: list, b: list, q: int) -> list:
    """Σημειακός πολλαπλασιασμός: επιστρέφει [a[i]*b[i] % q  για κάθε i]."""
    # TODO: return [x * y % q  for x, y in zip(a, b)]
    return [... for x, y in zip(a, b)]

def pointwise_add(a: list, b: list, q: int) -> list:
    """Σημειακή πρόσθεση: επιστρέφει [(a[i]+b[i]) % q  για κάθε i]."""
    # TODO: return [(x + y) % q  for x, y in zip(a, b)]
    return [... for x, y in zip(a, b)]

def matvec_ntt(M, v_ntt, q):
    """Πολλαπλασιασμός πίνακα-διανύσματος NTT (παρέχεται — μην τροποποιήσετε)."""
    k = len(M); n = len(M[0][0])
    result = []
    for row in range(k):
        acc = [0] * n
        for col in range(k):
            for i in range(n):
                acc[i] = (acc[i] + M[row][col][i] * v_ntt[col][i]) % q
        result.append(acc)
    return result

def keygen_t(A_hat, s_ntt, e_ntt, q):
    """Υπολογισμός t̂ = Â·ŝ + ê στον χώρο NTT.

    Για κάθε γραμμή:
        acc = [0]*n
        για κάθε στήλη:
            prod = pointwise_mul(A_hat[row][col], s_ntt[col], q)
            acc  = pointwise_add(acc, prod, q)
        t̂[row] = pointwise_add(acc, e_ntt[row], q)
    """
    k = len(A_hat); n = len(A_hat[0][0])
    t_hat = []
    for row in range(k):
        acc = [0] * n
        for col in range(k):
            # TODO: prod = pointwise_mul(A_hat[row][col], s_ntt[col], q)
            prod = ...
            # TODO: acc  = pointwise_add(acc, prod, q)
            acc  = ...
        # TODO: t_hat.append( pointwise_add(acc, e_ntt[row], q) )
        t_hat.append(...)
    return t_hat

t_hat = keygen_t(A,[S0_NTT,S1_NTT],[E0_NTT,E1_NTT],Q)
print(f"t̂[0]={t_hat[0]}  αναμενόμενο {T0}")
print(f"t̂[1]={t_hat[1]}  αναμενόμενο {T1}")
assert t_hat[0]==T0 and t_hat[1]==T1
print("\n✓ Το δημόσιο κλειδί t̂ συμφωνεί με τις καθολικές σταθερές.")


---
## Module 6: Ενθυλάκωση

### 6.1 Επισκόπηση

Ο Βόβος κατέχει το δημόσιο κλειδί `pk = (t̂, ρ)`. Επιλέγει μήνυμα `m` και νέο θόρυβο `r, e1, e2`,
και παράγει κρυπτοκείμενο `(u, v)` που μόνο η Αλίκη (με μυστικό κλειδί ŝ) μπορεί να αποκρυπτογραφήσει.

```
u = INTT(Â^T · r̂) + e1
v = INTT(t̂^T · r̂) + e2 + m̂
```

**Ανάστροφος:** `Â^T[i][j] = Â[j][i]` — γραμμές και στήλες του A εναλλάσσονται.

**Κωδικοποίηση μηνύματος:** κάθε bit του m κλιμακώνεται στο μέσο του Z_q:
```
bit 0 → 0 (near zero)
bit 1 → ⌊q/2⌋ = 8 (near the modulus midpoint)
```
Αυτό μεγιστοποιεί την απόσταση από λανθασμένη αποκωδικοποίηση.

### 6.2 Γιατί ο ανάστροφος;

Η αλγεβρική απαλοιφή στην αποενθυλάκωση απαιτεί:

```
v − ŝ^T · u = m̂ + small noise terms
```

Για να λειτουργήσει αυτό, ο Βόβος πρέπει να πολλαπλασιάζει με `Â^T` (όχι `Â`) κατά τον υπολογισμό του `u`.
Έτσι ο όρος `A·s` μέσα στο `t` απαλείφεται με τον όρο `A^T·r` μέσα στο `u`.

### 6.3 Υπολογισμός u — πλήρης ιχνηλάτηση για u[0]

```
Â^T row 0 = [Â[0][0], Â[1][0]] (column 0 of A, not row 0)

Â[0][0] ⊙ r̂[0]: [2×3, 11×5, 8×13, 1×7, 15×1, 3×6, 3×4, 6×3]
 = [6, 55→4, 104→2, 7, 15, 18→1, 12, 18→1]

Â[1][0] ⊙ r̂[1]: [7×3, 14×12, 3×13, 10×10, 5×1, 12×11, 1×4, 8×14]
 = [21→4, 168→15, 39→5, 100→15, 5, 132→13, 4, 112→10]

NTT sum: [10, 19→2, 7, 22→5, 20→3, 14, 16, 11]

INTT([10,2,7,5,3,14,16,11]) = [0, 15, 3, 11, 9, 16, 3, 4]

+ e1[0] = [0,0,0,1,0,0,0,0]: u[0] = [0, 15, 3, 12, 9, 16, 3, 4] ✓
```

---

### Άσκηση 6: Υλοποιήστε τη `encapsulate`

In [ ]:
def encode_message(msg: list, q: int) -> list:
    """Κωδικοποίηση κάθε bit: 0 → 0,  1 → q//2."""
    # TODO: return [bit * (q // 2)  for bit in msg]
    return [... for bit in msg]

def encapsulate(A_hat, t_hat, r, e1, e2, msg, roots, inv_roots, q, k, n_inv):
    """Ενθυλάκωση Kyber. Επιστρέφει (u, v).

    Βήματα:
      1. r_ntt = [ntt(rr, roots, q)  για rr in r]
      2. A_T[i][j] = A_hat[j][i]       (ανάστροφος)
      3. ATr = matvec_ntt(A_T, r_ntt, q)
      4. u[row] = intt(ATr[row]) + e1[row]  mod q
      5. tTr[i] = sum_col( t_hat[col][i] * r_ntt[col][i] ) mod q
      6. v = intt(tTr) + e2 + encode_message(msg)  mod q
    """
    n = len(r[0])

    # Βήμα 1 — NTT του r
    # TODO: r_ntt = [ntt(rr, roots, q)  για rr in r]
    r_ntt = [... for rr in r]

    # Βήμα 2 — Ανάστροφος A: A_T[i][j] = A_hat[j][i]
    # TODO: A_T = [[A_hat[j][i] for j in range(k)] for i in range(k)]
    A_T = [[... for j in range(k)] for i in range(k)]

    # Βήματα 3 & 4 — u (παρέχεται, χρησιμοποιεί r_ntt και A_T σας)
    ATr_ntt = matvec_ntt(A_T, r_ntt, q)
    u = []
    for row in range(k):
        u_row = intt(ATr_ntt[row], inv_roots, q, n_inv)
        u.append([(u_row[i] + e1[row][i]) % q for i in range(n)])

    # Βήματα 5 & 6 — v (παρέχεται, χρησιμοποιεί r_ntt σας)
    tTr_ntt = [sum(t_hat[col][i]*r_ntt[col][i] for col in range(k))%q for i in range(n)]
    v_pre = intt(tTr_ntt, inv_roots, q, n_inv)
    m_enc = encode_message(msg, q)
    v = [(v_pre[i] + e2[i] + m_enc[i]) % q for i in range(n)]
    return u, v

u_res,v_res = encapsulate(A,[T0,T1],[R0,R1],[E1_0,E1_1],E2,MSG,ROOTS,INV_ROOTS,Q,K,N_INV)
print(f"u[0]={u_res[0]}  αναμενόμενο {U0}")
print(f"u[1]={u_res[1]}  αναμενόμενο {U1}")
print(f"v   ={v_res}  αναμενόμενο {V}")
assert u_res[0]==U0 and u_res[1]==U1 and v_res==V
print("\n✓ Το κρυπτοκείμενο (u, v) συμφωνεί με τις καθολικές σταθερές.")


---
## Module 7: Αποενθυλάκωση

### 7.1 Υπολογισμός της Αλίκης

Η Αλίκη κατέχει το μυστικό κλειδί `ŝ` και λαμβάνει κρυπτοκείμενο `(u, v)`:

```
w = v − INTT(ŝ^T · NTT(u))
```

Στη συνέχεια κάθε συντελεστής του w αποκωδικοποιείται σε bit με βάση την **κυκλική απόσταση**:
- `dist_to_0 = min(w[i], q − w[i])`
- `dist_to_q/2 = min(|w[i]−8|, q − |w[i]−8|)`
- **bit = 1** if `dist_to_q/2 < dist_to_0`, else **bit = 0**

### 7.2 Γιατί λειτουργεί η άλγεβρα — η απαλοιφή

Αναπτύσσοντας αλγεβρικά τα v και u:
```
v = INTT(t̂^T·r̂) + e2 + m̂
 = INTT((Â·ŝ + ê)^T·r̂) + e2 + m̂
 = INTT(ŝ^T·Â^T·r̂) + INTT(ê^T·r̂) + e2 + m̂

ŝ^T·u = INTT(ŝ^T·(Â^T·r̂ + NTT(e1)))
 = INTT(ŝ^T·Â^T·r̂) + INTT(ŝ^T·NTT(e1))

w = v − ŝ^T·u
 = m̂ + INTT(ê^T·r̂) + e2 − INTT(ŝ^T·NTT(e1))
 └──────────────────────────────────────────┘
 NOISE TERMS ONLY
```

Οι όροι `INTT(ŝ^T·Â^T·r̂)` απαλείφονται ακριβώς.
Παραμένει μόνο θόρυβος — και επειδή τα s, e, r, e1, e2 έχουν μικρούς συντελεστές, ο θόρυβος είναι μικρός.

### 7.3 Προϋπολογισμός θορύβου

```
i w[i] m̂[i] noise = w[i]−m̂[i] |noise| < q/4=4.25?
0 10 8 +2 2 < 4.25 ✓
1 16 0 −1 (16−17=−1) 1 < 4.25 ✓
2 8 8 0 0 < 4.25 ✓
3 0 0 0 0 < 4.25 ✓
4 9 8 +1 1 < 4.25 ✓
5 1 0 +1 1 < 4.25 ✓
6 7 8 −1 1 < 4.25 ✓
7 0 0 0 0 < 4.25 ✓
Max noise = 2. All within budget. Decryption succeeds.
```

---

### Άσκηση 7: Υλοποιήστε τη `decapsulate`

In [ ]:
def decode_bit(coeff: int, q: int) -> int:
    """Αποκωδικοποίηση ενός συντελεστή σε bit.

    d0  = min(coeff, q-coeff)           απόσταση από 0
    dq2 = min(|coeff-q//2|,             απόσταση από q//2
               q-|coeff-q//2|)
    return 1 if dq2 < d0 else 0
    """
    d0  = min(coeff, q - coeff)
    dq2 = min(abs(coeff - q//2), q - abs(coeff - q//2))
    # TODO: return 1 if dq2 < d0 else 0
    return ...

def decapsulate(s_hat, u, v, roots, inv_roots, q, k, n_inv):
    """Αποενθυλάκωση Kyber. Επιστρέφει λίστα αποκωδικοποιημένων bits.

    Βήματα:
      1. u_ntt[i] = ntt(u[i], roots, q)
      2. stu[i]   = sum_j( s_hat[j][i] * u_ntt[j][i] ) mod q
      3. stu_n    = intt(stu, inv_roots, q, n_inv)
      4. w[i]     = (v[i] - stu_n[i]) % q
      5. return [decode_bit(c, q) for c in w]
    """
    n = len(u[0])
    # TODO: u_ntt = [ntt(uu, roots, q)  for uu in u]
    u_ntt   = [... for uu in u]
    stu_ntt = [sum(s_hat[j][i]*u_ntt[j][i] for j in range(k))%q for i in range(n)]
    # TODO: stu_n = intt(stu_ntt, inv_roots, q, n_inv)
    stu_n   = ...
    # TODO: w = [(v[i] - stu_n[i]) % q  for i in range(n)]
    w       = [... for i in range(n)]
    # TODO: return [decode_bit(c, q)  for c in w]
    return [... for c in w]

result = decapsulate([S0_NTT,S1_NTT],[U0,U1],V,ROOTS,INV_ROOTS,Q,K,N_INV)
print(f"Αποκωδικοποιημένο:  {result}")
print(f"Αρχικό: {MSG}")
assert result == DECODED
print("\n✓ Η αποενθυλάκωση συμφωνεί με τις καθολικές σταθερές.")


---
## Module 8: Πλήρης Δοκιμή Συστήματος

### 8.1 Η πλήρης αλυσίδα KeyGen → Ενθυλάκωση → Αποενθυλάκωση

Αυτό το κελί συγκεντρώνει τα πάντα σε μία εκτέλεση:

```
KeyGen:
 s_ntt, e_ntt = NTT(s), NTT(e)
 t_hat = A * s_ntt + e_ntt

Encapsulate:
 r_ntt = NTT(r)
 u = INTT(A^T * r_ntt) + e1
 v = INTT(t^T * r_ntt) + e2 + encode(m)

Decapsulate:
 w = v - INTT(s^T * NTT(u))
 m' = decode(w)
 assert m' == m
```

### 8.2 Γιατί ο Kyber είναι ασφαλής μετα-κβαντικά

Ο αλγόριθμος Shor σπάει RSA και ECDSA εκμεταλλευόμενος **περιοδικότητα** — η συνάρτηση `f(x) = aˣ mod N` επαναλαμβάνεται και ο Κβαντικός Μετασχηματισμός Fourier εντοπίζει την περίοδο.

Η συνάρτηση LWE `t = A·s + e` **δεν έχει περιοδικότητα**. Ο νέος θόρυβος `e` είναι κάθε φορά διαφορετικός, αποτρέποντας οποιαδήποτε περιοδική δομή. Ο QFT δεν βρίσκει τίποτα.

```
RSA: f(x) = 7^x mod 15 → [1,7,4,13,1,7,4,13,...] period = 4 (QFT breaks this)
LWE: t = A·s + e → output changes randomly no period (QFT finds nothing)
```

**Επίπεδα ασφάλειας:**
- Kyber-512: 2¹¹⁸ κβαντικές πράξεις για να σπάσει
- Kyber-768: 2¹⁷⁸ κβαντικές πράξεις για να σπάσει
- Kyber-1024: 2²⁵⁴ κβαντικές πράξεις για να σπάσει

---

### Άσκηση 8: Εκτελέστε την πλήρη αλυσίδα και επαληθεύστε από άκρο σε άκρο

In [ ]:
print("╔══════════════════════════════════════════════════════════╗")
print("║   CRYSTALS-Kyber  Πλήρης Δοκιμή Συστήματος              ║")
print(f"║   q={Q}  n={N}  k={K}  ETA={ETA}                                    ║")
print("╚══════════════════════════════════════════════════════════╝\n")

# Δημιουργία Κλειδιού
print("KeyGen")
s_ntt_kg = [ntt(S0, ROOTS, Q), ntt(S1, ROOTS, Q)]
e_ntt_kg = [ntt(E0, ROOTS, Q), ntt(E1, ROOTS, Q)]
t_hat_kg = keygen_t(A, s_ntt_kg, e_ntt_kg, Q)
assert t_hat_kg[0]==T0 and t_hat_kg[1]==T1
print(f"  t̂[0] = {t_hat_kg[0]}")
print(f"  t̂[1] = {t_hat_kg[1]}")
print("  ✓ Δημιουργία Κλειδιού ΟΚ\n")

# Ενθυλάκωση
print("Encapsulation")
print(f"  μήνυμα m = {MSG}")
u_enc, v_enc = encapsulate(
    A, t_hat_kg, [R0,R1], [E1_0,E1_1], E2, MSG,
    ROOTS, INV_ROOTS, Q, K, N_INV
)
assert u_enc[0]==U0 and u_enc[1]==U1 and v_enc==V
print(f"  u[0] = {u_enc[0]}")
print(f"  u[1] = {u_enc[1]}")
print(f"  v    = {v_enc}")
print("  ✓ Ενθυλάκωση ΟΚ\n")

# Αποενθυλάκωση
print("Decapsulation")
m_recovered = decapsulate(s_ntt_kg, u_enc, v_enc, ROOTS, INV_ROOTS, Q, K, N_INV)
assert m_recovered == MSG
print(f"  w       = {W}")
print(f"  αποκωδ. = {m_recovered}")
print(f"  αρχικό  = {MSG}")
print("  ✓ Αποενθυλάκωση ΟΚ\n")

# Αναφορά θορύβου
print("Noise Budget")
for i in range(N):
    noise = (W[i] - M_HAT[i]) % Q
    if noise > Q//2: noise -= Q
    bar = "█" * abs(noise) + "░" * (Q//4 - abs(noise))
    print(f"  coeff[{i}]  w={W[i]:2d}  m̂={M_HAT[i]}  noise={noise:+2d}  [{bar}]  < q/4={Q//4}")

print("\n╔══════════════════════════════════════════════════════════╗")
print("║    ΠΛΗΡΗΣ ΔΟΚΙΜΗ ΣΥΣΤΗΜΑΤΟΣ ΕΠΙΤΥΧΗΣ                  ║")
print("║  Δημιουργία Κλειδιού → Ενθυλάκωση → Αποενθυλάκωση  ✓   ║")
print("╚══════════════════════════════════════════════════════════╝")